# Collect licensed text for domain labelling: Pidgin, Igbo, Yoruba, Swahili, Kinyarwanda

Produces `output/<lang>_domain_candidates.xlsx` with the columns
**Language · Domain · Source · Text · Text_No_Punct**. The **Domain** column is left empty for you to fill
(each cell has a drop-down: finance, telecommunications, agriculture, health, law, others).

Only datasets published under a licence are used (Masakhane datasets, UD Naija, Swahili News, Rwandan Official Gazette,
XL-Sum, WURA, Wikipedia, FineWeb-2). Nothing is scraped from websites, and collections without a licence (e.g. IgboNLP) are not used.

**Run the cells top to bottom (Shift + Enter).** A normal CPU runtime is fine; one language takes about 5-15 minutes.

## 1. Settings: edit these

In [21]:
LANG = "yor"             # "pcm" Nigerian Pidgin, "ibo" Igbo, "yor" Yoruba, "swa" Swahili, "kin" Kinyarwanda
PER_DOMAIN = 60          # rows per domain (and for "others"); 50-70 recommended
COMMERCIAL_ONLY = False  # True = skip research-only licences (MasakhaNEWS, MAFAND, XL-Sum)
MAX_WORDS = 60           # longest passage (words); passages are whole sentences

## 2. Install packages

In [22]:
!pip -q install pandas pyarrow openpyxl huggingface_hub
print("ready")

ready


## 3. The collector script (no need to edit; it is saved as `collect_domain_text.py`)

In [23]:
%%writefile collect_domain_text.py
"""
collect_domain_text.py - collect openly licensed text for manual domain labelling in
Nigerian Pidgin (pcm), Igbo (ibo), Yoruba (yor), Swahili (swa) or Kinyarwanda (kin).

    python collect_domain_text.py --lang pcm                 # Nigerian Pidgin (also: ibo, yor, swa, kin)
    python collect_domain_text.py --lang ibo --per-domain 70 # Igbo, 70 candidates per domain
    python collect_domain_text.py --lang pcm --commercial-only   # only licences that allow commercial use

Output (in ./output/):
    <lang>_domain_candidates.xlsx   sheet "Data":    Language | Domain | Source | Text | Text_No_Punct
                                    (Domain is EMPTY on purpose - fill it in; the cell has a drop-down)
                                    sheet "Sources": every dataset used, its licence and how many rows it gave
    <lang>_selection_key.csv        the domain each row was selected for (kept out of the sheet so it does
                                    not influence your labels; compare with your labels afterwards)

How rows are chosen (no scraping of websites; only datasets published under a licence):
  1. download the licensed datasets for the language (GitHub + Hugging Face);
  2. split documents into short passages of whole sentences (default 8-60 words, ending in . ? !);
  3. keep passages that are clearly in the target language (marker words; English is dropped);
  4. remove duplicates (the same BBC article appears in several datasets);
  5. for each domain pick about --per-domain passages that match the domain: MasakhaNEWS topic
     labels first (health, business->finance), then keyword matches (2+ different keywords,
     clear lead over other domains), spread across sources; plus the same number of "others"
     (passages with no domain keywords);
  6. shuffle all rows so domains are mixed, and write the spreadsheet.

Licences: every source below states a licence. Sources marked commercial=False (MasakhaNEWS,
MAFAND, XL-Sum) allow research use only; run with --commercial-only to leave them out.
IgboNLP and similar collections that state no licence are deliberately NOT used.
"""
import argparse
import csv
import hashlib
import json
import os
import random
import re
import sys
import tarfile
import unicodedata
import urllib.request
from collections import Counter, defaultdict
from pathlib import Path

GH = "https://raw.githubusercontent.com"

# ---------------------------------------------------------------------------------------------
# 1. SOURCES - every one has a published licence. Edit/add here.
#    kind: tsv_masakhanews | json_mafand | conllu_ortho | xlsum_tar | parquet | jsonl
# ---------------------------------------------------------------------------------------------
SOURCES = {
    "pcm": [
        {"name": "MasakhaNEWS (pcm)", "kind": "tsv_masakhanews",
         "urls": [f"{GH}/masakhane-io/masakhane-news/main/data/pcm/{s}.tsv" for s in ("train", "dev", "test")],
         "license": "AFL-3.0 / CC BY-NC 4.0 (research use)", "commercial": False,
         "home": "https://github.com/masakhane-io/masakhane-news"},
        {"name": "MAFAND-MT (en-pcm)", "kind": "json_mafand", "side": "pcm",
         "urls": [f"{GH}/masakhane-io/lafand-mt/main/data/json_files/en-pcm/{s}.json" for s in ("train", "dev", "test")],
         "license": "CC BY-NC 4.0 (research use)", "commercial": False,
         "home": "https://github.com/masakhane-io/lafand-mt"},
        {"name": "UD Naija-NSC", "kind": "conllu_ortho",
         "urls": [f"{GH}/UniversalDependencies/UD_Naija-NSC/master/pcm_nsc-ud-{s}.conllu" for s in ("train", "dev", "test")],
         "license": "CC BY-SA 4.0", "commercial": True,
         "home": "https://github.com/UniversalDependencies/UD_Naija-NSC"},
        {"name": "XL-Sum (pidgin)", "kind": "xlsum_tar", "hf_repo": "csebuetnlp/xlsum",
         "hf_files": ["data/pidgin_XLSum_v2.0.tar.bz2"],
         "license": "CC BY-NC-SA 4.0 (research use)", "commercial": False,
         "home": "https://huggingface.co/datasets/csebuetnlp/xlsum"},
        {"name": "Wikipedia (pcm)", "kind": "parquet", "hf_repo": "wikimedia/wikipedia",
         "hf_files": ["20231101.pcm/train-00000-of-00001.parquet"],
         "license": "CC BY-SA 4.0", "commercial": True,
         "home": "https://huggingface.co/datasets/wikimedia/wikipedia"},
        {"name": "FineWeb-2 (pcm_Latn)", "kind": "parquet", "hf_repo": "HuggingFaceFW/fineweb-2",
         "hf_patterns": ["data/pcm_Latn/train/*"],
         "license": "ODC-By 1.0", "commercial": True,
         "home": "https://huggingface.co/datasets/HuggingFaceFW/fineweb-2"},
    ],
    "ibo": [
        {"name": "MasakhaNEWS (ibo)", "kind": "tsv_masakhanews",
         "urls": [f"{GH}/masakhane-io/masakhane-news/main/data/ibo/{s}.tsv" for s in ("train", "dev", "test")],
         "license": "AFL-3.0 / CC BY-NC 4.0 (research use)", "commercial": False,
         "home": "https://github.com/masakhane-io/masakhane-news"},
        {"name": "MAFAND-MT (en-ibo)", "kind": "json_mafand", "side": "ibo",
         "urls": [f"{GH}/masakhane-io/lafand-mt/main/data/json_files/en-ibo/{s}.json" for s in ("train", "dev", "test")],
         "license": "CC BY-NC 4.0 (research use)", "commercial": False,
         "home": "https://github.com/masakhane-io/lafand-mt"},
        {"name": "XL-Sum (igbo)", "kind": "xlsum_tar", "hf_repo": "csebuetnlp/xlsum",
         "hf_files": ["data/igbo_XLSum_v2.0.tar.bz2"],
         "license": "CC BY-NC-SA 4.0 (research use)", "commercial": False,
         "home": "https://huggingface.co/datasets/csebuetnlp/xlsum"},
        {"name": "WURA (ibo)", "kind": "jsonl", "hf_repo": "castorini/wura",
         "hf_files": ["documents-v1.0/train/ibo.jsonl"], "text_field": "content",
         "license": "Apache-2.0", "commercial": True,
         "home": "https://huggingface.co/datasets/castorini/wura"},
        {"name": "Wikipedia (ig)", "kind": "parquet", "hf_repo": "wikimedia/wikipedia",
         "hf_files": ["20231101.ig/train-00000-of-00001.parquet"],
         "license": "CC BY-SA 4.0", "commercial": True,
         "home": "https://huggingface.co/datasets/wikimedia/wikipedia"},
        {"name": "FineWeb-2 (ibo_Latn)", "kind": "parquet", "hf_repo": "HuggingFaceFW/fineweb-2",
         "hf_patterns": ["data/ibo_Latn/train/*"],
         "license": "ODC-By 1.0", "commercial": True,
         "home": "https://huggingface.co/datasets/HuggingFaceFW/fineweb-2"},
    ],
    "swa": [
        {"name": "MasakhaNEWS (swa)", "kind": "tsv_masakhanews",
         "urls": [f"{GH}/masakhane-io/masakhane-news/main/data/swa/{s}.tsv" for s in ("train", "dev", "test")],
         "license": "AFL-3.0 / CC BY-NC 4.0 (research use)", "commercial": False,
         "home": "https://github.com/masakhane-io/masakhane-news"},
        {"name": "Swahili News (Zenodo)", "kind": "parquet", "hf_repo": "community-datasets/swahili_news",
         "hf_files": ["swahili_news/train-00000-of-00001.parquet", "swahili_news/test-00000-of-00001.parquet"],
         "label_field": "label", "label_names": ["uchumi", "kitaifa", "michezo", "kimataifa", "burudani", "afya"],
         "license": "CC BY 4.0", "commercial": True,
         "home": "https://huggingface.co/datasets/community-datasets/swahili_news"},
        {"name": "MAFAND-MT (en-swa)", "kind": "json_mafand", "side": "swa",
         "urls": [f"{GH}/masakhane-io/lafand-mt/main/data/json_files/en-swa/{s}.json" for s in ("train", "dev", "test")],
         "license": "CC BY-NC 4.0 (research use)", "commercial": False,
         "home": "https://github.com/masakhane-io/lafand-mt"},
        {"name": "XL-Sum (swahili)", "kind": "xlsum_tar", "hf_repo": "csebuetnlp/xlsum",
         "hf_files": ["data/swahili_XLSum_v2.0.tar.bz2"],
         "license": "CC BY-NC-SA 4.0 (research use)", "commercial": False,
         "home": "https://huggingface.co/datasets/csebuetnlp/xlsum"},
        {"name": "Wikipedia (sw)", "kind": "parquet", "hf_repo": "wikimedia/wikipedia",
         "hf_files": ["20231101.sw/train-00000-of-00001.parquet"],
         "license": "CC BY-SA 4.0", "commercial": True,
         "home": "https://huggingface.co/datasets/wikimedia/wikipedia"},
        {"name": "FineWeb-2 (swh_Latn, test split)", "kind": "parquet", "hf_repo": "HuggingFaceFW/fineweb-2",
         "hf_patterns": ["data/swh_Latn/test/*"],      # the train file is 1.4 GB; the test split is enough here
         "license": "ODC-By 1.0", "commercial": True,
         "home": "https://huggingface.co/datasets/HuggingFaceFW/fineweb-2"},
    ],
    "yor": [
        {"name": "MasakhaNEWS (yor)", "kind": "tsv_masakhanews",
         "urls": [f"{GH}/masakhane-io/masakhane-news/main/data/yor/{s}.tsv" for s in ("train", "dev", "test")],
         "license": "AFL-3.0 / CC BY-NC 4.0 (research use)", "commercial": False,
         "home": "https://github.com/masakhane-io/masakhane-news"},
        {"name": "MAFAND-MT (en-yor)", "kind": "json_mafand", "side": "yor",
         "urls": [f"{GH}/masakhane-io/lafand-mt/main/data/json_files/en-yor/{s}.json" for s in ("train", "dev", "test")],
         "license": "CC BY-NC 4.0 (research use)", "commercial": False,
         "home": "https://github.com/masakhane-io/lafand-mt"},
        {"name": "XL-Sum (yoruba)", "kind": "xlsum_tar", "hf_repo": "csebuetnlp/xlsum",
         "hf_files": ["data/yoruba_XLSum_v2.0.tar.bz2"],
         "license": "CC BY-NC-SA 4.0 (research use)", "commercial": False,
         "home": "https://huggingface.co/datasets/csebuetnlp/xlsum"},
        {"name": "WURA (yor)", "kind": "jsonl", "hf_repo": "castorini/wura",
         "hf_files": ["documents-v1.0/train/yor.jsonl"], "text_field": "content",
         "license": "Apache-2.0", "commercial": True,
         "home": "https://huggingface.co/datasets/castorini/wura"},
        {"name": "Wikipedia (yo)", "kind": "parquet", "hf_repo": "wikimedia/wikipedia",
         "hf_patterns": ["20231101.yo/*"],
         "license": "CC BY-SA 4.0", "commercial": True,
         "home": "https://huggingface.co/datasets/wikimedia/wikipedia"},
        {"name": "FineWeb-2 (yor_Latn)", "kind": "parquet", "hf_repo": "HuggingFaceFW/fineweb-2",
         "hf_patterns": ["data/yor_Latn/train/*"],
         "license": "ODC-By 1.0", "commercial": True,
         "home": "https://huggingface.co/datasets/HuggingFaceFW/fineweb-2"},
    ],
    "kin": [
        {"name": "Rwandan Official Gazette (Digital Umuganda)", "kind": "tsv_parallel", "side": "kin",
         "hf_repo": "DigitalUmuganda/NMT_Rwandan-Gazette_parallel_data_en_kin",
         "hf_files": ["gazette_val.tsv", "gazette_test.tsv", "gazette_train.tsv"],
         "license": "CC BY 4.0", "commercial": True,
         "home": "https://huggingface.co/datasets/DigitalUmuganda/NMT_Rwandan-Gazette_parallel_data_en_kin"},
        {"name": "WURA (kin)", "kind": "jsonl", "hf_repo": "castorini/wura",
         "hf_files": ["documents-v1.0/train/kin.jsonl"], "text_field": "content",
         "license": "Apache-2.0", "commercial": True,
         "home": "https://huggingface.co/datasets/castorini/wura"},
        {"name": "Wikipedia (rw)", "kind": "parquet", "hf_repo": "wikimedia/wikipedia",
         "hf_files": ["20231101.rw/train-00000-of-00001.parquet"],
         "license": "CC BY-SA 4.0", "commercial": True,
         "home": "https://huggingface.co/datasets/wikimedia/wikipedia"},
        {"name": "FineWeb-2 (kin_Latn)", "kind": "parquet", "hf_repo": "HuggingFaceFW/fineweb-2",
         "hf_patterns": ["data/kin_Latn/train/*"],
         "license": "ODC-By 1.0", "commercial": True,
         "home": "https://huggingface.co/datasets/HuggingFaceFW/fineweb-2"},
    ],
}

LANG_NAMES = {"pcm": "Nigerian Pidgin", "ibo": "Igbo", "swa": "Swahili", "yor": "Yoruba",
              "kin": "Kinyarwanda"}
DOMAINS = ["finance", "telecommunications", "agriculture", "health", "law"]
OTHERS = "others"

# Dataset topic labels that map straight onto a domain (human labels, used first)
# (Swahili News labels: uchumi = economy -> finance, afya = health)
LABEL_MAP = {"health": "health", "business": "finance", "uchumi": "finance", "afya": "health"}

# ---------------------------------------------------------------------------------------------
# 2. KEYWORDS per language (matching ignores case and diacritics: ị=i, ọ=o, ụ=u, ṅ=n).
#    The non-target lists (politics, sports, ...) only compete, so e.g. an election story that
#    mentions one court is not taken as "law". Extend these freely - Igbo lists need a native check.
# ---------------------------------------------------------------------------------------------
KEYWORDS = {'ibo': {'agriculture': ['ọrụ ugbo', 'ugbo', 'ndị ọrụ ugbo', 'onye ọrụ ugbo', 'ihe ọkụkụ',
                         'owuwe ihe ubi', 'ubi', 'akpụ', 'osikapa', 'koko', 'fatịlaịza',
                         'fertilizer', 'anụ ụlọ', 'ehi', 'mkpụrụ', 'mkpụrụ osisi', 'ndị na-azụ anụ',
                         'herdsmen', 'agriculture', 'farmers', 'farm', 'harvest', 'crops',
                         'ọka ọkụkụ', 'ji ọkụkụ', 'ụkọ nri'],
         'education': ['agụmakwụkwọ', 'agụmagụ', 'ụlọ akwụkwọ', 'ụlọakwụkwọ', 'ụmụ akwụkwọ',
                       'ụmụakwụkwọ', 'nwa akwụkwọ', 'onye nkuzi', 'ndị nkuzi', 'nkuzi', 'mahadum',
                       'ule', 'waec', 'neco', 'jamb', 'asuu', 'ubec', 'praịmarị', 'sekọndrị',
                       'klas', 'klaasị', 'gụọ akwụkwọ', 'agụ akwụkwọ', 'school', 'schools',
                       'university', 'students', 'teachers', 'education', 'polytechnic', 'kọleji',
                       'sikọlashịp'],
         'entertainment': ['egwu', 'abụ', 'onye na-agụ egwu', 'nollywood', 'fim', 'ihe nkiri',
                           'bbnaija', 'big brother', 'davido', 'wizkid', 'burna boy', 'music',
                           'album', 'movie'],
         'finance': ['ego', 'naịra', 'naira', 'ụlọ akụ', 'ụlọakụ', 'akụnụba', 'azụmahịa', 'ụtụ isi',
                     'dọla', 'cbn', 'ọnụahịa', 'mmanụ ụgbọala', 'bank', 'economy', 'ego',
                     'akụ na ụba', 'mbinye ego', 'ụgwọ', 'ụtụ', 'ahịa ego', 'mbubata', 'mbupụ'],
         'health': ['ahụike', 'ahụ ike', 'ọrịa', 'ndị ọrịa', 'onye ọrịa', 'dọkịta', 'ndị dọkịta',
                    'ụlọ ọgwụ', 'ụlọọgwụ', 'ọgwụ', 'ọgwụgwọ', 'ọgwụ mgbochi', 'mgbochi ọrịa',
                    'nọọsụ', 'ndị nọọsụ', 'nje', 'ịba', 'ahụ ọkụ', 'ọrịa kansa', 'ọrịa shuga',
                    'ọrịa obi', 'ọbara', 'afọ ime', 'nwanyị dị ime', 'ọrịa na-efe efe', 'covid',
                    'covid-19', 'coronavirus', 'kọrọna', 'hiv', 'aids', 'lassa', 'kọlera',
                    'cholera', 'polio', 'ncdc', 'world health organization', 'hospital', 'doctor',
                    'doctors', 'vaccine', 'malaria', 'ahụike uche'],
         'law': ['ụlọikpe', 'ụlọ ikpe', 'ọkaikpe', 'ndị ọkaikpe', 'ọka iwu', 'onye ọka iwu',
                 'ndị ọka iwu', 'iwu', 'ikpe', 'ikpe ọmụma', 'mkpọrọ', 'ụlọ mkpọrọ', 'ụlọmkpọrọ',
                 'ebubo', 'boro ebubo', 'ndị uwe ojii', 'uwe ojii', 'uweojii', 'njide', 'jidere',
                 'nwụchiri', 'mpụ', 'omempụ', 'ndị omekome', 'efcc', 'icpc', 'supreme court',
                 'high court', 'court', 'judge', 'lawyer', 'bail', 'beelụ', 'iwu obodo',
                 'akwụkwọ iwu', 'ikike mmadụ', 'human rights'],
         'politics_government': ['ntuli aka', 'ntuliaka', 'inec', 'apc', 'pdp', 'apga', 'gọvanọ',
                                 'onyeisiala', 'onye isi ala', 'sineto', 'ndị omeiwu', 'gọọmentị',
                                 'ọchịchị', 'ndọrọ ndọrọ ọchịchị', 'minista', 'ipob', 'president',
                                 'governor', 'election'],
         'religion': ['chọọchị', 'ụka', 'fada', 'chineke', 'chukwu', 'jizọs', 'kraịst', 'ekpere',
                      'baịbụl', 'alakụba', 'ụlọ alakụba', 'pastọ', 'bishọp', 'church', 'mosque'],
         'sports': ['egwuregwu', 'bọọlụ', 'bọl', 'bọọlụ ụkwụ', 'agba bọọlụ', 'na-agba bọọlụ',
                    'goolu', 'asọmpi', 'super eagles', 'super falcons', 'flying eagles',
                    'golden eaglets', 'fifa', 'caf', 'afcon', 'world cup', 'iko mbaụwa',
                    'premier league', 'champions league', 'nff', 'npfl', 'klọb', 'klọọbụ',
                    'chelsea', 'arsenal', 'manchester', 'liverpool', 'barcelona', 'real madrid',
                    'olimpik', 'olympics', 'mgba', 'boxing', 'kọchị', 'coach', 'football', 'match',
                    'stadium', 'ama egwuregwu'],
         'telecommunications': ['mtn', 'glo', 'globacom', 'airtel', '9mobile', 'etisalat', 'ncc',
                                'ekwentị', 'ekwe ntị', 'ekwentị mkpanaaka', 'sim', 'kaadị sim',
                                'sim card', 'data', 'ịntanetị', 'intanet', 'internet', 'netwọk',
                                'network', 'airtime', 'kredit', 'rechaj', 'sms', 'ussd', '5g', '4g',
                                'broadband', 'nin', 'telekọm', 'telecom', 'telecoms', 'ozi ekwentị',
                                'smartphone']},
 'kin': {'agriculture': ['ubuhinzi', 'abahinzi', 'umuhinzi', 'ubworozi', 'aborozi', 'umworozi',
                         'umusaruro', 'ifumbire', 'imbuto', 'inka', 'amatungo', 'ibigori', 'ikawa',
                         'icyayi', 'umuceri', 'ibirayi', 'minagri', 'rab', 'kuhira', 'ibihingwa'],
         'education': ['amashuri', 'ishuri', 'abanyeshuri', 'umunyeshuri', 'abarimu', 'mwarimu',
                       'uburezi', 'kaminuza', 'ibizamini', 'reb'],
         'entertainment': ['umuziki', 'indirimbo', 'umuhanzi', 'abahanzi', 'filime', 'sinema',
                           'igitaramo'],
         'finance': ['amafaranga', 'banki', 'ubukungu', 'imisoro', 'umusoro', 'inguzanyo',
                     'ishoramari', 'abashoramari', 'ubucuruzi', 'abacuruzi', "ingengo y'imari",
                     'imari', 'bnr', 'rra', 'ibiciro', 'idolari', 'amadolari', 'konti', 'kuzigama',
                     'ubwizigame', 'umutungo'],
         'health': ['ubuzima', 'ibitaro', 'umuganga', 'abaganga', 'indwara', 'abarwayi', 'umurwayi',
                    'inkingo', 'urukingo', 'malariya', 'imiti', 'ivuriro', 'amavuriro', 'covid',
                    'covid-19', 'rbc', 'minisante', 'mituweli', "abajyanama b'ubuzima", 'virusi',
                    'sida', 'kanseri', 'diyabete'],
         'law': ['urukiko', 'inkiko', 'umucamanza', 'abacamanza', 'itegeko', 'amategeko',
                 'urubanza', 'imanza', 'ubushinjacyaha', 'rib', 'polisi', 'gereza', 'igifungo',
                 'icyaha', 'ibyaha', 'umwunganizi', 'avoka', 'abavoka', 'ubujurire',
                 'itegeko nshinga', 'ingingo ya', 'iteka', 'igazeti'],
         'politics_government': ['amatora', 'perezida', 'inteko ishinga amategeko', 'abadepite',
                                 'umudepite', 'minisitiri', 'guverinoma', 'ishyaka', 'fpr',
                                 'politiki', 'sena', 'abasenateri'],
         'religion': ['kiliziya', 'itorero', 'imana', 'idini', 'amasengesho', 'padiri',
                      'umupasiteri', 'umusigiti', 'abakirisitu'],
         'sports': ['umupira', "umupira w'amaguru", 'apr fc', 'rayon sports', 'amavubi', 'igitego',
                    'ibitego', 'shampiyona', 'umukino', 'imikino', 'abakinnyi', 'umukinnyi',
                    'ferwafa'],
         'telecommunications': ['telefoni', 'terefone', 'mtn', 'airtel', 'rura', 'interineti',
                                'murandasi', 'mobile money', 'momo', 'simukadi', 'sim card',
                                'itumanaho', 'ikoranabuhanga', 'data', '4g', '5g', 'sms',
                                'ubutumwa bugufi', 'internet', 'telecom']},
 'pcm': {'agriculture': ['farm', 'farms', 'farmer', 'farmers', 'farming', 'agric', 'agriculture',
                         'agricultural', 'crop', 'crops', 'harvest', 'rice', 'cassava', 'maize',
                         'corn', 'yam', 'cocoa', 'fertiliser', 'fertilizer', 'livestock', 'cattle',
                         'cow', 'cows', 'poultry', 'goat', 'herders', 'herdsmen', 'seed', 'seeds',
                         'irrigation', 'planting', 'plant', 'food security', 'food production',
                         'fish farming', 'tomato', 'tomatoes', 'pesticide', 'soil', 'grains'],
         'education': ['school', 'schools', 'student', 'students', 'teacher', 'teachers',
                       'university', 'universities', 'asuu', 'waec', 'jamb', 'exam', 'exams',
                       'education', 'lecturer', 'lecturers', 'pupils', 'classroom'],
         'entertainment': ['music', 'musician', 'song', 'album', 'nollywood', 'film', 'movie',
                           'actor', 'actress', 'bbnaija', 'big brother', 'celebrity', 'concert',
                           'artist', 'afrobeats', 'grammy', 'comedian', 'fashion'],
         'finance': ['naira', 'bank', 'banks', 'banking', 'money', 'loan', 'loans', 'cbn',
                     'central bank', 'inflation', 'economy', 'economic', 'tax', 'taxes', 'budget',
                     'dollar', 'dollars', 'investment', 'investors', 'stock', 'stocks',
                     'exchange rate', 'salary', 'salaries', 'debt', 'fintech', 'pos', 'account',
                     'accounts', 'business', 'businesses', 'price', 'prices', 'market',
                     'fuel price', 'subsidy', 'revenue', 'profit', 'interest rate', 'insurance',
                     'savings', 'cash', 'crypto', 'cryptocurrency', 'bitcoin', 'wages'],
         'health': ['hospital', 'hospitals', 'doctor', 'doctors', 'nurse', 'nurses', 'clinic',
                    'patient', 'patients', 'disease', 'diseases', 'sickness', 'sick', 'malaria',
                    'vaccine', 'vaccines', 'vaccination', 'covid', 'coronavirus', 'cholera',
                    'lassa', 'polio', 'hiv', 'aids', 'cancer', 'diabetes', 'pregnant', 'pregnancy',
                    'belle', 'medicine', 'medical', 'treatment', 'infection', 'virus', 'fever',
                    'ncdc', 'health', 'mental health', 'blood', 'surgery', 'drugs', 'tuberculosis',
                    'hepatitis', 'maternal', 'pikin health', 'immunisation', 'immunization',
                    'outbreak', 'symptoms', 'wellbeing'],
         'law': ['court', 'courts', 'judge', 'judges', 'justice', 'lawyer', 'lawyers', 'law',
                 'laws', 'legal', 'suit', 'lawsuit', 'trial', 'tribunal', 'verdict', 'judgment',
                 'judgement', 'constitution', 'constitutional', 'bail', 'prosecution', 'prosecutor',
                 'arraign', 'arraigned', 'jail', 'prison', 'sentence', 'sentenced', 'convicted',
                 'efcc', 'icpc', 'supreme court', 'high court', 'appeal', 'rights', 'human rights',
                 'attorney', 'legislation', 'bill', 'act', 'police', 'arrest', 'arrested',
                 'charges', 'charged'],
         'politics_government': ['election', 'elections', 'president', 'governor', 'senate',
                                 'senator', 'apc', 'pdp', 'inec', 'vote', 'votes', 'campaign',
                                 'minister', 'government', 'politics', 'political', 'party',
                                 'tinubu', 'buhari', 'atiku', 'assembly', 'lawmakers'],
         'religion': ['church', 'pastor', 'mosque', 'imam', 'god', 'allah', 'christian',
                      'christians', 'muslim', 'muslims', 'prayer', 'bible', 'quran', 'ramadan',
                      'easter', 'christmas', 'catholic', 'bishop'],
         'sports': ['football', 'match', 'goal', 'goals', 'super eagles', 'falcons', 'league',
                    'club', 'coach', 'striker', 'fifa', 'caf', 'afcon', 'world cup',
                    'premier league', 'player', 'players', 'tournament', 'olympics', 'athlete',
                    'boxing'],
         'telecommunications': ['mtn', 'glo', 'globacom', 'airtel', '9mobile', 'etisalat', 'ncc',
                                'telecom', 'telecoms', 'telecommunication', 'telecommunications',
                                'sim', 'sim card', 'sim cards', 'nin', 'airtime', 'recharge',
                                'data plan', 'data bundle', 'broadband', 'internet', '5g', '4g',
                                'ussd', 'sms', 'phone call', 'subscribers', 'mobile phone',
                                'smartphone', 'handset', 'fibre', 'fiber', 'spectrum',
                                'mobile network', 'network provider', 'data subscription',
                                'phone line']},
 'swa': {'agriculture': ['kilimo', 'wakulima', 'mkulima', 'mazao', 'zao', 'mbolea', 'mavuno',
                         'shamba', 'mashamba', 'mifugo', "ng'ombe", 'mahindi', 'kahawa', 'pamba',
                         'korosho', 'umwagiliaji', 'mbegu', 'uvuvi', 'wafugaji', 'mfugaji',
                         'pembejeo', 'viuatilifu', 'nzige'],
         'education': ['shule', 'wanafunzi', 'mwanafunzi', 'walimu', 'mwalimu', 'elimu',
                       'chuo kikuu', 'mtihani', 'mitihani', 'necta'],
         'entertainment': ['muziki', 'msanii', 'wasanii', 'wimbo', 'nyimbo', 'filamu',
                           'bongo flava', 'tamasha'],
         'finance': ['fedha', 'pesa', 'benki', 'uchumi', 'biashara', 'mkopo', 'mikopo', 'kodi',
                     'shilingi', 'dola', 'bei', 'mfumuko wa bei', 'uwekezaji', 'wawekezaji', 'hisa',
                     'soko la hisa', 'akiba', 'bajeti', 'mapato', 'deni', 'madeni', 'sarafu',
                     'benki kuu', 'tra', 'riba', 'mauzo', 'mtaji'],
         'health': ['afya', 'hospitali', 'daktari', 'madaktari', 'wagonjwa', 'mgonjwa', 'ugonjwa',
                    'magonjwa', 'malaria', 'chanjo', 'virusi', 'ukimwi', 'vvu', 'kifua kikuu',
                    'matibabu', 'dawa', 'kliniki', 'wauguzi', 'muuguzi', 'corona', 'covid',
                    'saratani', 'kisukari', 'zahanati', 'kituo cha afya', 'homa'],
         'law': ['mahakama', 'hakimu', 'jaji', 'majaji', 'sheria', 'wakili', 'mawakili', 'kesi',
                 'polisi', 'gereza', 'jela', 'mshtakiwa', 'washtakiwa', 'hukumu', 'rufaa', 'katiba',
                 'dhamana', 'upelelezi', 'kifungo', 'mashtaka', 'takukuru', 'mahakama kuu',
                 'mwendesha mashtaka', 'ushahidi'],
         'politics_government': ['uchaguzi', 'rais', 'wabunge', 'mbunge', 'bunge', 'chama', 'ccm',
                                 'chadema', 'serikali', 'waziri', 'kampeni', 'kura', 'siasa',
                                 'upinzani'],
         'religion': ['kanisa', 'msikiti', 'mungu', 'dini', 'askofu', 'padri', 'sala', 'mchungaji'],
         'sports': ['mpira', 'soka', 'timu', 'mchezo', 'michezo', 'bao', 'mabao', 'ligi', 'simba',
                    'yanga', 'kocha', 'mchezaji', 'wachezaji', 'taifa stars', 'fainali'],
         'telecommunications': ['simu', 'simu za mkononi', 'simu janja', 'intaneti',
                                'mtandao wa intaneti', 'mitandao ya simu', 'vodacom', 'airtel',
                                'safaricom', 'tigo', 'halotel', 'ttcl', 'tcra', 'laini ya simu',
                                'laini za simu', 'muda wa maongezi', 'kifurushi', 'vifurushi',
                                'data', 'bando', 'mawasiliano', 'minara', '4g', '5g', 'sms',
                                'ujumbe mfupi']},
 'yor': {'agriculture': ['àgbẹ̀', 'àwọn àgbẹ̀', 'iṣẹ́ àgbẹ̀', 'irè oko', 'ohun ọ̀gbìn', 'ọ̀gbìn',
                         'ajílẹ̀', 'ìkórè', 'ẹran ọ̀sìn', 'ọ̀sìn', 'màálù', 'àgbàdo', 'ìrẹsì',
                         'iṣu', 'ẹ̀gẹ́', 'darandaran', 'farmers', 'agriculture', 'fertilizer'],
         'education': ['ilé ẹ̀kọ́', 'ẹ̀kọ́', 'akẹ́kọ̀ọ́', 'olùkọ́', 'yunifásítì', 'ìdánwò', 'waec',
                       'jamb'],
         'entertainment': ['orin', 'olórin', 'fíìmù', 'nollywood', 'òṣèré', 'eré oníṣe'],
         'finance': ['owó', 'báǹkì', 'ilé ìfowópamọ́', 'ọrọ̀ ajé', 'owó orí', 'owó-orí', 'náírà',
                     'naira', 'dọ́là', 'ìsúná', 'òwò', 'oníṣòwò', 'gbèsè', 'cbn', 'ìnáwó', 'èrè',
                     'bank', 'economy'],
         'health': ['ìlera', 'ilé ìwòsàn', 'ilé-ìwòsàn', 'dókítà', 'àìsàn', 'àrùn', 'oògùn',
                    'abẹ́rẹ́ àjẹsára', 'àjẹsára', 'ibà', 'kòkòrò àrùn', 'covid', 'kòrónà',
                    'nọ́ọ̀sì', 'aláìsàn', 'ìtọ́jú', 'hospital', 'malaria', 'vaccine'],
         'law': ['ilé ẹjọ́', 'ilé-ẹjọ́', 'adájọ́', 'agbẹjọ́rò', 'òfin', 'ẹjọ́', 'ọlọ́pàá', 'ẹ̀wọ̀n',
                 'ọgbà ẹ̀wọ̀n', 'ìdájọ́', 'ẹ̀sùn', 'olùjẹ́jọ́', 'beeli', 'ìwé òfin', 'efcc',
                 'court', 'police'],
         'politics_government': ['ìdìbò', 'ààrẹ', 'gómìnà', 'ẹgbẹ́ òṣèlú', 'òṣèlú', 'apc', 'pdp',
                                 'inec', 'ìjọba', 'aṣòfin', 'sẹ́nétọ̀'],
         'religion': ['ṣọ́ọ̀ṣì', 'mọ́ṣáláṣí', 'ọlọ́run', 'àdúrà', 'kristẹni', 'mùsùlùmí', 'pásítọ̀',
                      'àlùfáà', 'ìmáàmù'],
         'sports': ['bọ́ọ̀lù', 'eré ìdárayá', 'super eagles', 'góòlù', 'ìdíje', 'agbábọ́ọ̀lù',
                    'fifa', 'premier league'],
         'telecommunications': ['fóònù', 'ẹ̀rọ ìbánisọ̀rọ̀', 'ẹ̀rọ alágbèéká', 'íńtánẹ́ẹ̀tì', 'mtn',
                                'glo', 'airtel', '9mobile', 'ncc', 'sim', 'káàdì sim', 'dátà',
                                'data', 'nẹ́tíwọ̀ọ̀kì', 'airtime', 'káàdì ìpè', 'sms', '4g', '5g',
                                'internet', 'telecom']}}

# Frequent function words of each language (no diacritics) for the language check.
LANG_MARKERS = {k: set(v) for k, v in {'ibo': ['ahu', 'ala', 'anyi', 'bu', 'di', 'dika', 'ebe', 'emere', 'ga', 'gi', 'ha', 'ihe', 'ihi',
         'ime', 'ka', 'kama', 'kwa', 'kwuru', 'maka', 'mana', 'maobu', 'mere', 'mgbe', 'na',
         'nakwa', 'ndi', 'niile', 'nile', 'nke', 'nwanyi', 'nwere', 'nwoke', 'nyere', 'o', 'obodo',
         'oge', 'onye', 'otu', 'ozi', 'ozo', 'ruo', 'site', 'ugbu', 'umu', 'ya'],
 'kin': ['abantu', 'aho', 'ari', 'ariko', 'bityo', 'buri', 'bya', 'byinshi', 'byo', 'cya', 'cyane',
         'igihe', 'iyi', 'kandi', 'ko', 'ku', 'kubera', 'kugira', 'kuko', 'kuri', 'mu', 'na',
         'ndetse', 'ngo', 'ni', 'nta', 'ntabwo', 'nyuma', 'rya', 'ubwo', 'uko', 'uyu'],
 'pcm': ['abi', 'afta', 'anoda', 'becos', 'belle', 'bin', 'chop', 'comot', 'dat', 'dem', 'dey',
         'di', 'dia', 'dis', 'don', 'enta', 'fit', 'gree', 'im', 'kontri', 'na', 'naim', 'neva',
         'oda', 'odas', 'ontop', 'ova', 'pikin', 'pikin dem', 'pipo', 'sabi', 'sef', 'sey', 'sotay',
         'sotey', 'tink', 'tins', 'tok', 'tori', 'una', 'wahala', 'wan', 'wen', 'wetin', 'wey',
         'wit'],
 'swa': ['alisema', 'ambao', 'ambayo', 'baada', 'cha', 'hadi', 'hao', 'hata', 'hii', 'hili', 'hiyo',
         'huo', 'ili', 'kama', 'katika', 'kutoka', 'kuwa', 'kwa', 'kwamba', 'kwenye', 'la',
         'lakini', 'mwaka', 'na', 'ni', 'pia', 'sana', 'vya', 'wa', 'wakati', 'wake', 'wao', 'ya',
         'yake', 'za', 'zaidi'],
 'yor': ['ati', 'awon', 'bi', 'eyi', 'fun', 'gege', 'je', 'kan', 'ko', 'lati', 'lo', 'maa', 'naa',
         'ni', 'nigba', 'ninu', 'nitori', 'o', 'ohun', 'oro', 'pe', 'se', 'si', 'sibe', 'sugbon',
         'tabi', 'ti', 'tun', 'wa', 'won', 'yii', 'yoo']}.items()}
ENG_MARKERS = {"the", "is", "are", "was", "were", "has", "have", "been", "their", "they", "which",
               "this", "that", "these", "those", "would", "could", "should", "its", "an", "whose",
               "whom", "does", "being"}
LANG_EVIDENCE_CHARS = {"pcm": "", "ibo": "ịụṅỊỤṄ", "yor": "ẹṣẸṢ", "swa": "", "kin": ""}


# ---------------------------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------------------------
def fold(t: str) -> str:
    t = unicodedata.normalize("NFD", t.lower().replace("’", "'"))
    return "".join(c for c in t if not unicodedata.combining(c))


WORD_RE = re.compile(r"[^\W_]+(?:['\-][^\W_]+|(?<=\d)[.,]\d+)*")
SENT_RE = re.compile(r'(?<=[.!?])["\')\]]?\s+(?=["\'(\[]?[A-Z0-9À-ɏḀ-ỿ])')
TERMINAL_RE = re.compile(r'[.!?]["\')\]]*$')


def words(t):
    return WORD_RE.findall(t)


def clean(t: str) -> str:
    t = unicodedata.normalize("NFC", t or "")
    t = re.sub(r"https?://\S+|www\.\S+", " ", t)
    t = t.translate(str.maketrans({"“": '"', "”": '"', "‘": "'", "’": "'",
                                   "–": "-", "—": "-", "…": "...", "​": ""}))
    t = re.sub(r"[ \t]+([.,!?;:])", r"\1", t)
    return re.sub(r"\s+", " ", t).strip()


KEEP_SYMBOLS = set("£$€₦%&+")


def text_no_punct(text: str) -> str:
    """Lower-case and remove punctuation. Kept: apostrophes/hyphens inside words (don't,
    two-time), separators inside numbers (2.5, 15,000), currency/percent signs, & and +."""
    t = unicodedata.normalize("NFC", text)
    seps = re.findall(r"(?<=\d)[.,](?=\d)", t)
    t = re.sub(r"(?<=\d)[.,](?=\d)", "", t)
    t = re.sub(r"(?<=\w)'(?=\w)", "", t)
    t = re.sub(r"(?<=\w)-(?=\w)", "", t)
    out = []
    for ch in t:
        if ch in "" or ch in KEEP_SYMBOLS or unicodedata.category(ch)[0] in "LNM" or ch.isspace():
            out.append(ch)
        else:
            out.append(" ")
    s, j = [], 0
    for ch in "".join(out):
        if ch == "":
            s.append(seps[j]); j += 1
        else:
            s.append({"": "-", "": "'"}.get(ch, ch))
    return re.sub(r"\s+", " ", "".join(s)).strip().lower()


def compile_keywords(lang):
    pats = {}
    for dom, ws in KEYWORDS[lang].items():
        alts = sorted({re.escape(fold(w)) for w in ws}, key=len, reverse=True)
        pats[dom] = re.compile(r"(?<![\w])(" + "|".join(alts) + r")(?![\w])")
    return pats


def keyword_domain(text, pats, min_hits, min_distinct, margin=1.5):
    low = fold(text)
    found = {d: p.findall(low) for d, p in pats.items()}
    ranked = sorted(found.items(), key=lambda kv: len(kv[1]), reverse=True)
    (best, hits), (_, nxt) = ranked[0], ranked[1]
    total_target = sum(len(v) for d, v in found.items() if d in DOMAINS)
    ok = len(hits) >= min_hits and len(set(hits)) >= min_distinct and len(hits) >= margin * max(len(nxt), 1)
    return (best if ok else None), total_target


def is_language(text, lang):
    toks = [w.lower() for w in words(text)]
    if len(toks) < 6:
        return False
    markers = LANG_MARKERS[lang]
    ev = LANG_EVIDENCE_CHARS.get(lang, "")
    tgt = sum((fold(t) in markers) or (bool(ev) and any(c in t for c in ev)) for t in toks) / len(toks)
    eng = sum(t in ENG_MARKERS for t in toks) / len(toks)
    return tgt >= 0.04 and tgt >= eng


def passages(doc: str, min_words: int, max_words: int):
    """Groups of whole consecutive sentences, each ending in . ? or !"""
    for para in re.split(r"\n+", doc):
        sents = [s.strip() for s in SENT_RE.split(clean(para)) if s.strip()]
        cur, n = [], 0
        for s in sents:
            k = len(words(s))
            if cur and n + k > max_words:
                yield " ".join(cur)
                cur, n = [], 0
            if k > max_words:            # one very long sentence: skip it
                continue
            cur.append(s)
            n += k
        if cur:
            yield " ".join(cur)


def norm_key(t):
    return hashlib.md5(re.sub(r"[\W_]+", "", fold(t)).encode()).hexdigest()


# ---------------------------------------------------------------------------------------------
# Download + read
# ---------------------------------------------------------------------------------------------
def fetch(url, dest: Path):
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not dest.exists():
        with urllib.request.urlopen(url, timeout=120) as r:
            dest.write_bytes(r.read())
    return dest


def fetch_hf(src, cache: Path):
    from huggingface_hub import hf_hub_download, snapshot_download
    out = cache / src["hf_repo"].replace("/", "__")
    paths = []
    for f in src.get("hf_files", []):
        paths.append(Path(hf_hub_download(repo_id=src["hf_repo"], repo_type="dataset", filename=f,
                                          local_dir=str(out))))
    if src.get("hf_patterns"):
        snapshot_download(repo_id=src["hf_repo"], repo_type="dataset", local_dir=str(out),
                          allow_patterns=src["hf_patterns"])
        for pat in src["hf_patterns"]:
            paths += sorted(out.glob(pat))
    return paths


def read_docs(src, cache: Path, max_docs: int):
    """Yield (text, label) for one source."""
    kind = src["kind"]
    n = 0
    if "urls" in src:
        files = [fetch(u, cache / src["name"].replace(" ", "_") / u.rsplit("/", 1)[1]) for u in src["urls"]]
    else:
        files = fetch_hf(src, cache)
    for f in files:
        if kind == "tsv_masakhanews":
            import pandas as pd
            df = pd.read_csv(f, sep="\t", dtype=str, keep_default_na=False)
            df.columns = [c.lower() for c in df.columns]
            for _, r in df.iterrows():
                yield r.get("text", ""), (r.get("category") or r.get("label") or "").strip().lower()
                n += 1
        elif kind == "json_mafand":
            for line in open(f, encoding="utf-8"):
                if line.strip():
                    yield json.loads(line)["translation"].get(src["side"], ""), ""
                    n += 1
        elif kind == "conllu_ortho":
            doc, cur = [], None
            for line in open(f, encoding="utf-8"):
                if line.startswith("# sent_id"):
                    d = line.split("=", 1)[1].strip().rsplit("__", 1)[0]
                    if cur is not None and d != cur and doc:
                        yield " ".join(doc), ""
                        n += 1
                        doc = []
                    cur = d
                elif line.startswith("# text_ortho"):
                    doc.append(line.split("=", 1)[1].strip())
            if doc:
                yield " ".join(doc), ""
        elif kind == "xlsum_tar":
            with tarfile.open(f, "r:bz2") as tf:
                for m in tf.getmembers():
                    if m.name.endswith(".jsonl"):
                        for line in tf.extractfile(m):
                            if line.strip():
                                yield json.loads(line).get("text", ""), ""
                                n += 1
        elif kind == "parquet":
            import pyarrow.parquet as pq
            pf = pq.ParquetFile(f)
            lab = src.get("label_field")
            cols = ["text"] + ([lab] if lab and lab in pf.schema_arrow.names else [])
            names = src.get("label_names", [])
            for batch in pf.iter_batches(batch_size=1000, columns=cols):
                texts = batch.column("text").to_pylist()
                labels = batch.column(lab).to_pylist() if len(cols) > 1 else [None] * len(texts)
                for t, l in zip(texts, labels):
                    if isinstance(l, int) and 0 <= l < len(names):
                        l = names[l]
                    yield t or "", str(l or "").lower()
                    n += 1
                    if n >= max_docs:
                        return
        elif kind == "tsv_parallel":
            import pandas as pd
            df = pd.read_csv(f, sep="\t", dtype=str, keep_default_na=False, quoting=3, on_bad_lines="skip")
            col = src["side"] if src["side"] in df.columns else df.columns[0]
            for t in df[col]:
                yield t or "", ""
                n += 1
                if n >= max_docs:
                    return
        elif kind == "jsonl":
            for line in open(f, encoding="utf-8"):
                if line.strip():
                    d = json.loads(line)
                    yield d.get(src.get("text_field", "text")) or "", ""
                    n += 1
                    if n >= max_docs:
                        return
        if n >= max_docs:
            return


# ---------------------------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------------------------
def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--lang", choices=sorted(SOURCES), default="pcm")
    ap.add_argument("--per-domain", type=int, default=60, help="rows per domain (and for 'others')")
    ap.add_argument("--min-words", type=int, default=8)
    ap.add_argument("--max-words", type=int, default=60)
    ap.add_argument("--max-docs", type=int, default=30000, help="max documents read per source")
    ap.add_argument("--commercial-only", action="store_true", help="skip research-only (NC) licences")
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--out", default="output")
    args = ap.parse_args()
    rng = random.Random(args.seed)
    lang = args.lang
    out = Path(args.out)
    out.mkdir(exist_ok=True)
    cache = Path("cache") / lang
    pats = compile_keywords(lang)

    pools = defaultdict(lambda: defaultdict(list))     # domain -> source -> [(text, how)]
    seen, src_info, src_stats = set(), [], Counter()
    for src in SOURCES[lang]:
        if args.commercial_only and not src["commercial"]:
            print(f"[skip] {src['name']}: research-only licence ({src['license']})")
            continue
        print(f"[read] {src['name']} ...", flush=True)
        n_pass = 0
        try:
            for doc, label in read_docs(src, cache, args.max_docs):
                for p in passages(doc, args.min_words, args.max_words):
                    nw = len(words(p))
                    if nw < args.min_words or not TERMINAL_RE.search(p) or not is_language(p, lang):
                        continue
                    k = norm_key(p)
                    if k in seen:
                        continue
                    seen.add(k)
                    n_pass += 1
                    dom, target_hits = keyword_domain(p, pats, min_hits=2, min_distinct=2)
                    if label in LABEL_MAP:
                        # human topic label: keep only passages that also look like the domain
                        if dom == LABEL_MAP[label] or (dom is None and target_hits >= 1):
                            pools[LABEL_MAP[label]][src["name"]].append((p, "dataset label"))
                    elif dom in DOMAINS:
                        pools[dom][src["name"]].append((p, "keywords"))
                    elif target_hits == 0:
                        pools[OTHERS][src["name"]].append((p, "no domain keywords"))
        except Exception as e:                      # one failing source should not stop the rest
            print(f"    FAILED: {type(e).__name__}: {e}")
            continue
        src_info.append(src)
        print(f"    {n_pass:,} usable passages")

    # pick per domain, spreading across sources (round-robin), dataset labels first
    rows, short = [], []
    for dom in DOMAINS + [OTHERS]:
        by_src = pools.get(dom, {})
        for lst in by_src.values():
            rng.shuffle(lst)
            lst.sort(key=lambda x: 0 if x[1] == "dataset label" else 1)
        picked, names = [], sorted(by_src)
        while len(picked) < args.per_domain and any(by_src[s] for s in names):
            for s in names:
                if by_src[s] and len(picked) < args.per_domain:
                    t, how = by_src[s].pop(0)
                    picked.append((dom, s, t, how))
        if len(picked) < args.per_domain:
            short.append(f"{dom}: {len(picked)}")
        rows += picked
    rng.shuffle(rows)
    if not rows:
        sys.exit("No rows found: every source failed or gave no usable passages. Check your internet / "
                 "Hugging Face access (in Colab: from huggingface_hub import notebook_login; notebook_login()).")

    # ---- write the spreadsheet
    from openpyxl import Workbook
    from openpyxl.styles import Alignment, Font, PatternFill
    from openpyxl.worksheet.datavalidation import DataValidation
    wb = Workbook()
    ws = wb.active
    ws.title = "Data"
    ws.append(["Language", "Domain", "Source", "Text", "Text_No_Punct"])
    for dom, s, t, how in rows:
        ws.append([LANG_NAMES[lang], None, s, t, text_no_punct(t)])
    f, fb = Font(name="Arial", size=10), Font(name="Arial", size=10, bold=True)
    for row in ws.iter_rows():
        for c in row:
            c.font = f
            c.alignment = Alignment(wrap_text=c.column >= 4, vertical="top")
    for c in ws[1]:
        c.font = fb
        c.fill = PatternFill("solid", fgColor="D9E2F3")
    for col, w in zip("ABCDE", (16, 20, 22, 80, 80)):
        ws.column_dimensions[col].width = w
    dv = DataValidation(type="list", formula1='"' + ",".join(DOMAINS + [OTHERS]) + '"', allow_blank=True)
    ws.add_data_validation(dv)
    dv.add(f"B2:B{len(rows) + 1}")
    ws.freeze_panes = "A2"

    used = Counter(s for _, s, _, _ in rows)
    s2 = wb.create_sheet("Sources")
    s2.append(["Source", "Licence", "Commercial use", "Rows in Data", "Link"])
    for src in src_info:
        s2.append([src["name"], src["license"], "yes" if src["commercial"] else "no (research only)",
                   used.get(src["name"], 0), src["home"]])
    s2.append([])
    s2.append(["Domain", "Empty on purpose: choose finance, telecommunications, agriculture, health, law or others"])
    s2.append(["Text_No_Punct", "Text lower-cased with punctuation removed (apostrophes/hyphens inside words "
                                "and separators inside numbers kept)"])
    for row in s2.iter_rows():
        for c in row:
            c.font = f
    for c in s2[1]:
        c.font = fb
    for col, w in zip("ABCDE", (24, 40, 20, 14, 60)):
        s2.column_dimensions[col].width = w
    xlsx = out / f"{lang}_domain_candidates.xlsx"
    wb.save(xlsx)

    with open(out / f"{lang}_selection_key.csv", "w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(["row_in_Data_sheet", "selected_for_domain", "selected_by", "Source"])
        for i, (dom, s, t, how) in enumerate(rows, start=2):
            w.writerow([i, dom, how, s])

    print(f"\nWrote {len(rows)} rows -> {xlsx}")
    print("Rows per (selected) domain:", dict(Counter(d for d, *_ in rows)))
    print("Rows per source:", dict(used))
    if short:
        print("Fewer than --per-domain found for:", ", ".join(short),
              "-> add keywords for that domain or raise --max-docs.")


if __name__ == "__main__":
    main()


Overwriting collect_domain_text.py


## 4. Run it

In [24]:
flag = "--commercial-only" if COMMERCIAL_ONLY else ""
!python collect_domain_text.py --lang {LANG} --per-domain {PER_DOMAIN} --max-words {MAX_WORDS} {flag}

[read] MasakhaNEWS (yor) ...
    17,621 usable passages
[read] MAFAND-MT (en-yor) ...
    8,016 usable passages
[read] XL-Sum (yoruba) ...

data/yoruba_XLSum_v2.0.tar.bz2: downloading bytes:  59% 2.96M/5.05M [00:01<00:01, 1.62MB/s]
data/yoruba_XLSum_v2.0.tar.bz2: downloading bytes: 100% 5.05M/5.05M [00:01<00:00, 2.74MB/s,  480kB/s  ]
data/yoruba_XLSum_v2.0.tar.bz2: reconstructing file: 100% 5.05M/5.05M [00:01<00:00, 2.73MB/s,  480kB/s  ]
    55,576 usable passages
[read] WURA (yor) ...

documents-v1.0/train/yor.jsonl: downloading bytes:  28% 54.2M/196M [00:02<00:02, 51.5MB/s, 3.16MB/s  ]
documents-v1.0/train/yor.jsonl: downloading bytes:  52% 102M/196M [00:02<00:01, 73.5MB/s, 8.00MB/s  ] 
documents-v1.0/train/yor.jsonl: downloading bytes: 100% 103M/103M [00:03<00:00, 34.0MB/s, 9.14MB/s  ]
documents-v1.0/train/yor.jsonl: reconstructing file: 100% 196M/196M [00:03<00:00, 64.7MB/s, 17.8MB/s  ]
    291,340 usable passages
[read] Wikipedia (yo) ...
Reconstructing (incomplete total...): |   

## 5. Preview and download

In [25]:
import pandas as pd
df = pd.read_excel(f"output/{LANG}_domain_candidates.xlsx", sheet_name="Data")
print(df.shape)
display(df.head(10))
display(pd.read_excel(f"output/{LANG}_domain_candidates.xlsx", sheet_name="Sources"))
try:
    from google.colab import files
    files.download(f"output/{LANG}_domain_candidates.xlsx")
    files.download(f"output/{LANG}_selection_key.csv")
except ImportError:
    print("files are in ./output")

(360, 5)


,Language,Domain,Source,Text,Text_No_Punct
0,Yoruba,NaN,Wikipedia (yo),"Macy Gray (oruko abiso Natalie McIntyre, 6 Sep...",macy gray oruko abiso natalie mcintyre 6 septe...
1,Yoruba,NaN,FineWeb-2 (yor_Latn),"Arabinrin Folake, eni ogota odun ti gbe oko re...",arabinrin folake eni ogota odun ti gbe oko re ...
2,Yoruba,NaN,WURA (yor),Lati šii foonu rẹ pẹlu Dr.Fone - SIM Ṣii silẹ ...,lati šii foonu rẹ pẹlu dr fone sim ṣii silẹ se...
3,Yoruba,NaN,MasakhaNEWS (yor),"Bakan naa ni isọri gold, ẹgbẹrun lọna ọgọta na...",bakan naa ni isọri gold ẹgbẹrun lọna ọgọta nai...
4,Yoruba,NaN,XL-Sum (yoruba),Awọm musulumi lo po ju lọ lorile-ede Malaysia ...,awọm musulumi lo po ju lọ lorile-ede malaysia ...
5,Yoruba,NaN,XL-Sum (yoruba),Lara awọn to wa ni ipade naa ni awọn alaga NCC...,lara awọn to wa ni ipade naa ni awọn alaga ncc...
6,Yoruba,NaN,XL-Sum (yoruba),Ileeṣẹ ti awọn oṣiṣẹ naa n ṣiṣẹ fun ni wọn pe ...,ileeṣẹ ti awọn oṣiṣẹ naa n ṣiṣẹ fun ni wọn pe ...
7,Yoruba,NaN,WURA (yor),"Adajọ ni, ""Ju gbogbo rẹ lọ, awọn ọdaran mejeej...",adajọ ni ju gbogbo rẹ lọ awọn ọdaran mejeeji y...
8,Yoruba,NaN,MasakhaNEWS (yor),Saaju la ti mu iroyin wa fun yin pe lonii Ọjọr...,saaju la ti mu iroyin wa fun yin pe lonii ọjọr...
9,Yoruba,NaN,MAFAND-MT (en-yor),Dáramólá ní wọn ti se ọkọ̀ òfúrufú NAF C-130H ...,dáramólá ní wọn ti se ọkọ̀ òfúrufú naf c-130h ...


,Source,Licence,Commercial use,Rows in Data,Link
0,MasakhaNEWS (yor),AFL-3.0 / CC BY-NC 4.0 (research use),no (research only),48.0,https://github.com/masakhane-io/masakhane-news
1,MAFAND-MT (en-yor),CC BY-NC 4.0 (research use),no (research only),47.0,https://github.com/masakhane-io/lafand-mt
2,XL-Sum (yoruba),CC BY-NC-SA 4.0 (research use),no (research only),68.0,https://huggingface.co/datasets/csebuetnlp/xlsum
3,WURA (yor),Apache-2.0,yes,70.0,https://huggingface.co/datasets/castorini/wura
4,Wikipedia (yo),CC BY-SA 4.0,yes,57.0,https://huggingface.co/datasets/wikimedia/wiki...
5,FineWeb-2 (yor_Latn),ODC-By 1.0,yes,70.0,https://huggingface.co/datasets/HuggingFaceFW/...
6,NaN,NaN,NaN,NaN,NaN
7,Domain,"Empty on purpose: choose finance, telecommunic...",NaN,NaN,NaN
8,Text_No_Punct,Text lower-cased with punctuation removed (apo...,NaN,NaN,NaN


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**`<lang>_selection_key.csv`** says which domain each row was *selected for* (by MasakhaNEWS topic label or
keywords). Keep it closed while you label, then compare it with your labels if you want to see how well the
selection worked.